# ⚡ LLM 推理速览 — 5 分钟理解推理在做什么

**目标**：如果你对 LLM 内部机制还比较陌生，这篇用最简洁的方式让你理解推理的全貌——token、自回归生成、推理服务的核心指标。

读完这篇，你会带着正确的 mental model 进入后续的 Transformer 推理基础和量化原理章节。

## 1. Token：LLM 的"字母表"

LLM 不直接处理文本，而是处理 **token**——把文本切成的一个个小片段。

```
输入文本: "Hello, world! 你好世界"
         ↓ tokenizer
Tokens:  ["Hello", ",", " world", "!", " 你好", "世界"]
         ↓ tokenizer.encode()
IDs:     [9906, 11, 1917, 0, 5012, 2234]
         ↓ model (一堆矩阵运算)
Logits:  [..., 0.02, 0.15, ..., 0.73, ...]  ← 每个 token 一个"得分"
         ↓ sampling (选最高分 → greedy)
输出:    9906 → "Hello"
```

关键点：
- **Tokenizer** 是 LLM 的"输入法"——把人类文字翻译成模型认识的数字
- **Vocabulary**（词表）通常有 32k ~ 128k 个 token，每个有唯一 ID
- Token ≠ 单词：中文一个 token 可能是一个字或词，英文可能是字母组合

## 2. 自回归生成：为什么是逐 token 输出？

LLM 推理是**自回归**（autoregressive）的——每次只生成**一个 token**，然后把这个 token 拼接回输入，再生成下一个：

```
Step 0: "请解释相对论"  → 模型运算 → token: "相"
Step 1: "请解释相对论相"  → 模型运算 → token: "对"
Step 2: "请解释相对论相对"  → 模型运算 → token: "论"
Step 3: "请解释相对论相对论" → 模型运算 → token: "是"
...
Step N: "请解释相对论相对论是..." → 模型运算 → token: <EOS>（结束符）
```

**为什么要这样？**
因为语言是有顺序依赖的——每个新词的意思取决于前面说了什么。模型必须在看到前文后才"知道"下一个词应该是什么。

这也是为什么推理慢的根本原因：生成 100 个 token 需要**跑 100 次模型**。

## 3. 推理服务的本质

把 `model.generate()` 包装成一个**高并发的 HTTP API 服务**：

```
用户请求
  POST /v1/chat/completions
  {"messages": [{"role": "user", "content": "你好"}]}
       ↓
┌──────────────────────────────────┐
│         推理服务 (Server)          │
│  ┌──────────────────────────────┐ │
│  │     调度器 (Scheduler)        │ │  ← 决定先处理谁、怎么排队
│  └──────────┬───────────────────┘ │
│             ↓                     │
│  ┌──────────────────────────────┐ │
│  │    推理引擎 (Engine)           │ │  ← 真正跑模型的地方
│  │   • vLLM / TensorRT-LLM /    │ │
│  │     llama.cpp / SGLang ...   │ │
│  └──────────────────────────────┘ │
└──────────────────────────────────┘
       ↓
  流式响应 (SSE)
  data: {"choices": [{"delta": {"content": "你"}}]}
  data: {"choices": [{"delta": {"content": "好"}}]}
  data: {"choices": [{"delta": {"content": "！"}}]}
  data: [DONE]
```

推理框架的核心工作就是**把有限的 GPU 资源（显存、算力）高效分配给大量并发请求**。

## 4. 延迟的三重奏

推理服务有两个核心延迟指标：

```
用户发送 "写一首关于春天的诗"
  │
  ├─ 收到第一个 token ──── TTFT (Time To First Token)
  │   "春"                    首 token 延迟
  │
  ├─ "风"
  ├─ "拂"                  TPOT (Time Per Output Token)
  ├─ "过"                      每个 token 之间的间隔
  ├─ ...
  │
  └─ 最后 token <EOS> ──── Total Latency（总延迟）

另外还有：
• TPOT 中位数 (P50): 50% 的 token 间隔 ≤ 这个值 → 感受"流畅度"
• TPOT P99:      99% 的 token 间隔 ≤ 这个值 → 反映"抖动"最坏情况
• Throughput:    每秒能处理多少个 token（总吞吐）
```

### 核心矛盾

| 目标 | 手段 | 副作用 |
|------|------|--------|
| 降低 TTFT | 让新请求插队（preemption）、增加 GPU 并行度 | 可能增加已有请求的延迟 |
| 提高吞吐 | 批处理更多请求（larger batch） | 单个请求变慢（排队） |
| 降低 P99 | 公平调度、预留资源 | 总体利用率下降 |

**推理框架的每一个设计决策都是在这些矛盾中做权衡**。理解了这一点，就能理解为什么会有 PagedAttention、Continuous Batching、RadixAttention 等等——它们都是针对不同矛盾场景的优化方案。

## 5. 下一步

现在你理解了推理服务的基本概念：

- ✅ Token 和 tokenizer 的工作方式
- ✅ 自回归生成——逐 token、跑 N 次模型
- ✅ 推理服务的三层结构（调度器 → 引擎 → 模型）
- ✅ TTFT / TPOT / 吞吐的核心矛盾

接下来进入 **Transformer 推理基础**，理解为什么一次推理（尤其是 Attention）会成为计算和显存瓶颈——这是所有推理框架优化的出发点。

---

## 补充：一个最小化的推理 demo

下面这段代码展示了 `model.generate()` 在做什么——不需要 GPU，只是一个概念演示。

In [ ]:
# 概念演示：自回归生成的简化模型

import random

class ToyLLM:
    """
    一个假 LLM，随机返回下一个 token，但展示了推理的控制流
    """
    def __init__(self, vocab_size=100, max_len=20):
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.eos = 0  # token 0 = <EOS>
    
    def _fake_forward(self, input_ids):
        """假装跑模型 → 返回 logits (随机得分)"""
        scores = [random.random() for _ in range(self.vocab_size)]
        scores[self.eos] = 0.05  # EOS 概率较低
        return scores
    
    def generate(self, prompt_ids, max_new_tokens=10):
        """自回归生成循环"""
        generated = list(prompt_ids)  # 从 prompt 开始
        
        for step in range(max_new_tokens):
            # 1. 模型前向传播: 输入全部序列 → 得到每个位置的概率
            logits = self._fake_forward(generated)
            
            # 2. 取最后一个位置的预测 → 采样得到下一个 token
            next_token = max(range(len(logits)), key=lambda i: logits[i])
            
            # 3. 拼回去
            generated.append(next_token)
            
            # 4. 如果遇到结束符，停止
            if next_token == self.eos:
                print(f"   → Step {step}: <EOS>, 停止生成")
                break
            else:
                print(f"   → Step {step}: token={next_token}, 序列长度={len(generated)}")
        
        return generated


print("=== 自回归生成演示 ===\n")
model = ToyLLM(vocab_size=100)
prompt = [10, 20, 30]  # 假 prompt tokens
print(f"Prompt tokens: {prompt}")
print(f"开始生成 (max_new_tokens=10):\n")
output = model.generate(prompt, max_new_tokens=10)
print(f"\n最终输出: {output} (共 {len(output)} 个 token, prompt {len(prompt)} + 生成 {len(output)-len(prompt)})")
print(f"\n关键观察: 生成 {len(output)-len(prompt)} 个新 token 需要跑 {len(output)-len(prompt)} 次模型")

## 补充：TTFT 和 TPOT 的计算示例

```python
# 假设一个推理请求的时间线
timeline = [
    (0.00, "收到请求"),
    (0.05, "请求出队，开始 prefill"),
    (0.35, "prefill 完成"),          # TTFT = 0.35s
    (0.35, "生成 token 1: '春'"),    # TPOT[1] = 0.35 - 0.35 = 0.00s (第一个不计)
    (0.42, "生成 token 2: '风'"),    # TPOT[2] = 0.42 - 0.35 = 0.07s
    (0.48, "生成 token 3: '拂'"),    # TPOT[3] = 0.48 - 0.42 = 0.06s
    (0.55, "生成 token 4: '过'"),    # TPOT[4] = 0.55 - 0.48 = 0.07s
]

# 指标计算:
TTFT = 0.35  # 收到请求 → 第一个 token
TPOTs = [0.07, 0.06, 0.07]  # [token2间隔, token3间隔, token4间隔]
TPOT_P50 = sorted(TPOTs)[len(TPOTs)//2]  # 中位数: 0.07s
Total_Latency = 0.55  # 总延迟
Throughput = 4 / 0.55  # 4 tokens / 0.55s ≈ 7.3 tokens/s
```
